# A first look at Gaia data

In this notebook we work with real data from Gaia Data Release 3 (DR3, released June 2022).
We will:

1. Query the Gaia archive and load a table of all well-measured stars within 50 parsecs
2. Use `pandas` to explore the table
3. Find our nearest neighbours, including a few famous ones
4. Make a map of the sky
5. Make a Hertzsprung-Russell diagram

As before, **Try it** cells have blanks (`____`) next to a `# your code here` comment. Later cells
often use the result of a **Try it** cell, so get each one working before moving on.

Each new piece of `pandas` code is explained the first time it appears, and there's a short
summary at the end.

In [ ]:
# Setup. If the data files aren't here (for example on Google Colab), download them.
# On Colab, also install astroquery.
import os, sys, subprocess, urllib.request

DATA_DIR = "data/"
DATA_URL = "https://raw.githubusercontent.com/vedantchandra/srmp-gaia/main/data/"
os.makedirs(DATA_DIR, exist_ok=True)
for name in ["gaia_50pc.csv"]:
    if not os.path.exists(DATA_DIR + name):
        print("downloading", name)
        urllib.request.urlretrieve(DATA_URL + name, DATA_DIR + name)

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroquery"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord

## 1. Getting the data

Gaia's catalogs live in the **Gaia archive** (https://gea.esac.esa.int/archive/). The main table,
`gaiadr3.gaia_source`, has one row for each of 1.8 billion sources and about 150 columns.
That's far too much to download, so we ask the archive for just the rows and columns we want.

The archive understands a language called **ADQL**, which is a version of SQL (a standard
language for querying databases). A query looks like this:

```sql
SELECT <columns> FROM <table> WHERE <conditions>
```

The `astroquery` package sends the query for us and gives back a table. Here's a tiny one: all
stars with a parallax above 300 mas, so closer than about 3.3 pc.

In [ ]:
from astroquery.gaia import Gaia

query = '''
SELECT source_id, ra, dec, parallax, phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE parallax > 300
'''

try:
    result = Gaia.launch_job(query).get_results()
    print(result)
except Exception as e:
    print("Could not reach the Gaia archive. That's OK, we'll use saved data below.")
    print(e)

If that worked, you just searched a catalog of 1.8 billion stars, and found about ten.
(You may also see a message about "breaking changes" in the archive. That's a notice for
programmers, not an error. You can ignore it.)

Some of the columns:

| Column | Meaning | Units |
|---|---|---|
| `source_id` | Unique ID number for each source | |
| `ra`, `dec` | Right ascension and declination: position on the sky | degrees |
| `parallax` | Parallax $\varpi$ | mas |
| `parallax_error` | Uncertainty on the parallax | mas |
| `pmra`, `pmdec` | Proper motion: how fast the star moves across the sky, in the RA and Dec directions. Written $\mu_{\alpha*}$ and $\mu_\delta$ in equations | mas/yr |
| `phot_g_mean_mag` | Brightness in Gaia's G band (a broad optical filter). Magnitudes are backwards: bigger = fainter | mag |
| `bp_rp` | Colour: brightness difference between Gaia's blue (BP) and red (RP) filters. Bigger = redder (cooler) | mag |
| `ruwe` | "Renormalised unit weight error": how far the star's measured positions are from the best simple model of a single star moving in a straight line (plus parallax). About 1 is normal. Much bigger than 1 means the simple model doesn't fit | |

The full list is in the Gaia DR3 documentation
(https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html).

A star that wobbles because of an orbiting companion doesn't fit the single-star model well, so it
gets a high RUWE. We'll use that later to look for binaries.

### 1.1 The 50 parsec sample

Now the real query: every star with parallax above 20 mas (so closer than 50 pc), whose parallax
is measured to better than 10% (`parallax_over_error > 10`), and which has a measured colour.

We also ask for a few quality columns (`phot_bp_rp_excess_factor` and the ones after it).
We'll see why we need them in section 5.

This returns about 45,000 stars and takes a minute or so. A copy of the result is saved in the
`data/` folder, so by default we load that instead. Set `QUERY_ARCHIVE = True` to run the query
yourself.

In [ ]:
QUERY_ARCHIVE = False

query_50pc = '''
SELECT source_id, ra, dec, parallax, parallax_error, pmra, pmdec,
       phot_g_mean_mag, bp_rp, ruwe,
       phot_bp_rp_excess_factor, visibility_periods_used, astrometric_excess_noise,
       phot_bp_mean_flux_over_error, phot_rp_mean_flux_over_error
FROM gaiadr3.gaia_source
WHERE parallax > 20
  AND parallax_over_error > 10
  AND bp_rp IS NOT NULL
'''

if QUERY_ARCHIVE:
    job = Gaia.launch_job_async(query_50pc)
    stars = job.get_results().to_pandas()
else:
    stars = pd.read_csv(DATA_DIR + "gaia_50pc.csv")

print(len(stars), "stars")

## 2. pandas: working with tables

`stars` is a pandas **DataFrame**: a table with named columns, a bit like a spreadsheet. Each
column is basically a numpy array, so everything from the Python basics notebook works on it.

(In the `describe()` output below, the `source_id` numbers are shown in scientific notation
because they are huge. The IDs themselves are fine.)

In [ ]:
stars.head()      # first 5 rows

In [ ]:
stars.describe()  # summary statistics of every column

Get a column with `stars["column_name"]`. Make a new column by assigning to a new name.
Select rows with a boolean mask, just like numpy: `stars[stars["parallax"] > 100]`.

**Try it.** Add a column called `dist_pc` with the distance in parsecs. Then find how many stars
are within 10 pc.

In [ ]:
stars["dist_pc"] = ____    # your code here

within_10pc = stars[____]    # your code here
print(len(within_10pc), "stars within 10 pc")

A quick check: the volume of a sphere grows as $r^3$. If stars are spread evenly around us, there
should be $(50/10)^3 = 125$ times more stars within 50 pc than within 10 pc. Is that roughly
true? Before you look, think about what could make it come out higher or lower. (For example: are
there stars too bright, or too faint, for Gaia to measure? Would that matter more at 10 pc or at
50 pc?) We'll come back to this number.

In [ ]:
print(len(stars) / len(within_10pc))

### 2.1 Sorting

`sort_values` sorts the table by a column. Here are the 10 nearest stars in our sample.

In [ ]:
stars.sort_values("dist_pc").head(10)

The nearest is Proxima Centauri, at 1.30 pc. Notice who is **missing**: Alpha Centauri A and B,
the two bright stars that Proxima orbits. They are about as close (1.34 pc) but are so bright
that they saturate Gaia's detectors, and DR3 doesn't give them a good parallax. The same goes for
Sirius and the other very brightest stars in the sky.

Every catalog has gaps like this. A big part of doing research with a survey is knowing what it
can and can't see.

## 3. Some famous neighbours

Here are the Gaia `source_id`s for some well-known nearby stars:

In [ ]:
famous = {
    "Proxima Centauri": 5853498713190525696,
    "Barnard's Star":   4472832130942575872,
    "Wolf 359":         3864972938605115520,
    "Sirius B":         2947050466531873024,
    "Epsilon Eridani":  5164707970261890560,
    "Tau Ceti":         2452378776434477184,
    "40 Eridani A":     3195919528989223040,
    "40 Eridani B":     3195919254111315712,
    "40 Eridani C":     3195919254111314816,
    "61 Cygni A":       1872046609345556480,
    "61 Cygni B":       1872046574983497216,
}

for name, sid in famous.items():
    row = stars[stars["source_id"] == sid].iloc[0]
    print(f"{name:18s} d = {row['dist_pc']:5.2f} pc   G = {row['phot_g_mean_mag']:5.2f}   BP-RP = {row['bp_rp']:5.2f}")

(`.iloc[0]` picks the first row of the selection. There's exactly one row for each ID.)

A bit about each:

- **Proxima Centauri**: the nearest star to the Sun. A small, cool red dwarf with at least one
  roughly Earth-mass planet in its *habitable zone* (the range of distances from a star where a
  planet could have liquid water on its surface).
- **Barnard's Star**: the star that moves fastest across our sky. Four small planets were found
  around it in 2024 and 2025.
- **Wolf 359**: one of the closest stars, but so dim (a red dwarf about 1/1000 as luminous as
  the Sun) that you need a telescope to see it.
- **Sirius B**: Sirius is the brightest star in the night sky. Sirius B is its tiny companion,
  a **white dwarf**: the leftover core of a dead star, about the size of the Earth. Sirius A is
  too bright for Gaia DR3, but Sirius B is in.
- **Epsilon Eridani**: a young star a bit cooler than the Sun, with a giant planet and a ring of
  dust and rocky debris around it, a bit like our Solar System's asteroid and Kuiper belts.
- **Tau Ceti**: one of the nearest Sun-like stars. In the science-fiction novel
  *Project Hail Mary* (Andy Weir, 2021), it is the star the spaceship *Hail Mary* travels to.
- **40 Eridani**: a triple system. 40 Eridani B was the first white dwarf ever identified, and C is
  a red dwarf. In *Project Hail Mary*, 40 Eridani A is the star orbited by Erid, the home planet
  of the alien character Rocky. (It's also the home of the planet Vulcan in *Star Trek*.)
- **61 Cygni**: a pair of orange dwarfs. In 1838 Friedrich Bessel measured its parallax,
  about 0.3 arcseconds. This was the first time anyone measured the distance to a star.

**Try it.** Barnard's Star is supposed to be the fastest-moving star on the sky. Check this.

1. Compute the total proper motion $\mu = \sqrt{\mu_{\alpha*}^2 + \mu_\delta^2}$ (that is,
   from the `pmra` and `pmdec` columns) for every star, and store it in a new column `pm_total`
   (units: mas/yr).
2. Sort by `pm_total` to find the fastest stars.
3. Is the fastest one Barnard's Star? Compare its `source_id` with `famous["Barnard's Star"]`.
4. How long would it take Barnard's Star to move across the width of the full Moon (about 0.5
   degrees, or 1,800,000 mas)?

In [ ]:
stars["pm_total"] = ____    # your code here

fastest = stars.sort_values(____, ascending=False).head(5)    # your code here
print(fastest[["source_id", "dist_pc", "pm_total", "phot_g_mean_mag"]])

top = fastest.iloc[0]          # .iloc[0] = the first row
print("Fastest is Barnard's Star:", top["source_id"] == ____)    # your code here

barnard_pm = top["pm_total"]
print(f"Years to cross the Moon: {____:.0f}")    # your code here

About 10 arcseconds per year. That sounds fast, but it still takes Barnard's Star about 170 years
to move one Moon-width. Gaia can measure motions 100,000 times smaller than this.

## 4. A map of the sky

`ra` and `dec` are like longitude and latitude on the sky. Let's just plot them.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(stars["ra"], stars["dec"], s=1, alpha=0.3)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_xlim(360, 0)    # astronomers draw RA increasing to the left (as you see it looking up)
plt.show()

This squashes the sphere of the sky onto a rectangle, like a world map that makes Greenland look
huge. A better option is a map projection that keeps areas correct. matplotlib has a few built in.
They want angles in **radians**, with longitude between $-\pi$ and $\pi$.

It's also more natural to use **Galactic coordinates** for this. Galactic longitude $l$ and
latitude $b$ are defined so that the plane of the Milky Way is at $b = 0$ and the Galactic
center is at $l = 0$. `astropy.coordinates.SkyCoord` converts between coordinate systems.

In [ ]:
coords = SkyCoord(ra=stars["ra"].values * u.deg, dec=stars["dec"].values * u.deg, frame="icrs")
gal = coords.galactic

stars["l"] = gal.l.wrap_at(180 * u.deg).deg   # longitude from -180 to 180 instead of 0 to 360
stars["b"] = gal.b.deg

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(np.radians(stars["l"]), np.radians(stars["b"]), s=1, alpha=0.3)
ax.grid(True)
ax.set_title("Stars within 50 pc, Galactic coordinates")
plt.show()

(Like RA, Galactic longitude is usually drawn increasing to the left. matplotlib's projections
draw it increasing to the right. We'll ignore that for now.)

The stars are spread fairly evenly over the whole sky. That might surprise you if you've seen
photos of the Milky Way, which is a band across the sky. Most stars in the Milky Way's disk are
within about 300 pc of its middle plane, and the Sun is near that plane. So within 50 pc we're deep
inside the disk, and the stars around us are in every direction.
Go further out and they concentrate toward $b = 0$.

Two things don't fit that picture, though:

- There are dense clumps near the center of the map ($l \approx 0$, $b \approx 0$). That's the
  direction of the Galactic center, thousands of parsecs away. Why would there be extra stars
  *within 50 pc* in that direction? Keep this in mind for section 5.
- There are faint stripes and arcs. Those come from **Gaia's scanning pattern**: some parts of
  the sky were observed more often than others, so more stars there pass our
  `parallax_over_error > 10` cut.

**Try it.** Add the famous stars to the map as larger red points, with their names next to them.
(`ax.text(x, y, "words")` writes text at position `x, y`. Add a small number to `x` and `y`, like
0.03, so the label doesn't sit right on top of the point. Remember these positions are in radians, not
degrees.)

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(np.radians(stars["l"]), np.radians(stars["b"]), s=1, alpha=0.2, color="gray")

for name, sid in famous.items():
    row = stars[stars["source_id"] == sid].iloc[0]
    x, y = np.radians(row[____]), np.radians(row[____])    # your code here
    ax.scatter(x, y, s=30, color="red")
    ax.text(____, ____, name, fontsize=8)    # your code here

ax.grid(True)
plt.show()

## 5. The Hertzsprung-Russell diagram

The most useful plot in stellar astronomy puts a star's **colour** (which tells you its
temperature) on the x-axis and how much light it really gives off on the y-axis. It's called a
Hertzsprung-Russell (HR) diagram, or a colour-magnitude diagram.

**Magnitudes.** (This uses logarithms: $\log_{10}(x)$ asks "10 to what power gives $x$?")
Astronomers measure brightness in *magnitudes*. If a star sends us a flux $F$
(energy per second per square meter), its magnitude is

$$ m = -2.5 \log_{10} F + \text{constant}. $$

The minus sign is why magnitudes are backwards: brighter stars have *smaller* magnitudes. A
difference of 5 magnitudes is a factor of exactly 100 in flux. (This odd system goes back to the
ancient Greeks, who called the brightest stars "first magnitude".)

The apparent magnitude $G$ tells us how bright a star *looks* in Gaia's G band, which depends on
its distance. To compare stars fairly we use the **absolute magnitude** $M_G$: how bright the star
would look from a standard distance of 10 pc. Flux falls off as $1/d^2$, so moving a star from
distance $d$ to 10 pc changes its magnitude by $2.5 \log_{10}(d^2/10^2) = 5 \log_{10}(d/10)$:

$$ M_G = G - 5 \log_{10}\!\left(\frac{d}{10\,\mathrm{pc}}\right) = G + 5 \log_{10}\!\left(\frac{\varpi}{100\,\mathrm{mas}}\right) $$

(Both forms are the same, because $d = 1000/\varpi$.) We're ignoring dust between us and the
stars, which would make them look fainter. Within 50 pc there's very little dust, so that's fine.

**Try it.** Add a column `abs_G` with the absolute magnitude. Then plot `bp_rp` on the x-axis
and `abs_G` on the y-axis. Because magnitudes are backwards, flip the y-axis so that bright stars
are at the top (`ax.invert_yaxis()`).

In [ ]:
stars["abs_G"] = ____    # your code here

fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(____, ____, s=1, alpha=0.3, color="k")    # your code here
____    # your code here
ax.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax.set_ylabel(r"$M_G$ [mag]")
plt.show()

Some of the structure here is real, and some of it isn't. Things you should see:

- **The main sequence**: the long diagonal band from top left (hot, bright, blue) to bottom
  right (cool, faint, red). These are stars burning hydrogen in their cores, like the Sun. Most
  nearby stars are red dwarfs at the lower right, because small stars are much more common than
  big ones.
- **White dwarfs**: the separate, thinner sequence at the lower left. Hot but very faint,
  because they are tiny (about the size of the Earth).
- **Giants**: a few stars at the top right. Cool but very luminous, because they are huge.
  There are only a few, because the giant phase of a star's life is short.

And one thing that shouldn't be there: a big **cloud of points below the main sequence**, around
$M_G \approx 17$. No known kind of star lives there in such numbers.

### 5.1 Something is wrong

Let's find out what these objects are. Select the stars in the cloud and compare them to the
rest.

**Try it.** Select stars with `abs_G > 15.5` and `bp_rp` between 0.8 and 3.5, and call them
`weird`. How many are there? Then compare the median of a few columns between `weird` and
everything else: `phot_g_mean_mag`, `visibility_periods_used` (how many separate times Gaia
looked at the star), and `phot_bp_mean_flux_over_error` (how precisely the blue brightness was
measured). Finally, plot where the `weird` stars are on the sky, using the `l` and `b` columns
you made earlier (converted with `np.radians`, as before).

Reminder: to combine conditions, put each one in parentheses and join them with `&`, like
`(stars["bp_rp"] > 0.8) & (stars["bp_rp"] < 3.5)`. Writing `0.8 < stars["bp_rp"] < 3.5` gives
an error about "truth value of a Series is ambiguous".

In [ ]:
weird_mask = (stars["abs_G"] > 15.5) & (____) & (____)    # your code here
weird = stars[weird_mask]
normal = stars[~weird_mask]
print(len(weird), "weird objects")

for col in ["phot_g_mean_mag", "visibility_periods_used", "phot_bp_mean_flux_over_error"]:
    print(f"{col:30s} weird: {weird[col].median():8.2f}   normal: {____:8.2f}")    # your code here

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(____, ____, s=1, color="C3")    # your code here
ax.grid(True)
plt.show()

The weird objects are very faint (around $G = 20$, close to Gaia's limit), were observed fewer
times, have poorly measured colours, and are concentrated toward the Galactic center and the
Galactic plane. These are the most crowded parts of the sky.

What's going on: in crowded fields, faint stars have close neighbours, and Gaia sometimes mixes
up measurements of one star with another. That can produce a large parallax by accident.
These are distant stars with **wrong parallaxes**, not nearby stars. Their colours are wrong too,
because light from the neighbours leaks into the BP and RP measurements.

We deal with this using **quality cuts**: conditions that remove sources with unreliable
measurements.

The main one uses `phot_bp_rp_excess_factor`, which compares the total flux in BP and RP to the G
flux. For a clean, isolated source it follows a known relation with colour. Contaminated sources
have too much flux in BP and RP. The cut below comes from the Gaia team's DR2 HR diagram paper
(Gaia Collaboration, Babusiaux et al. 2018). We also require enough separate visits, and well
measured BP and RP fluxes.

In [ ]:
excess = stars["phot_bp_rp_excess_factor"]
color = stars["bp_rp"]

good = (
    (excess < 1.3 + 0.06 * color**2)
    & (excess > 1.0 + 0.015 * color**2)
    & (stars["visibility_periods_used"] > 10)
    & (stars["phot_bp_mean_flux_over_error"] > 10)
    & (stars["phot_rp_mean_flux_over_error"] > 10)
)
clean = stars[good]
print(f"kept {len(clean)} of {len(stars)} stars")

fig, ax = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
ax[0].scatter(stars["bp_rp"], stars["abs_G"], s=1, alpha=0.3, color="k")
ax[0].set_title("all")
ax[1].scatter(clean["bp_rp"], clean["abs_G"], s=1, alpha=0.3, color="k")
ax[1].set_title("after quality cuts")
ax[0].invert_yaxis()
for a in ax:
    a.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax[0].set_ylabel(r"$M_G$ [mag]")
plt.show()

Much better. Now go back to the star count test from section 2. **Try it.** With the clean sample,
what is the ratio of stars within 50 pc to stars within 10 pc? How does it compare to 125? Be careful
to count the stars within 10 pc **in `clean`**, not the `within_10pc` table from before.

In [ ]:
ratio = len(clean) / len(clean[____])    # your code here
print(ratio)

Cuts are never perfect. They always throw away some real stars along with the junk, and you
should check what you lost. Which of our famous stars fail the cuts? `~good` means "not good", and
`.isin(...)` checks whether each value is in a list:

In [ ]:
fails = stars[~good & stars["source_id"].isin(famous.values())]
fails[["source_id", "dist_pc", "phot_g_mean_mag", "bp_rp", "phot_bp_rp_excess_factor"]]

It's Sirius B. Why would Sirius B, of all stars, have contaminated colours? (Think about what's
right next to it on the sky.)

Here's why this matters for our project. The BP and RP measurements collect light from a small
window around each star, about 3.5 by 2 arcseconds. Light from any star close by on the sky,
including a binary companion, gets mixed in. So this cut also tends to remove some close binaries.

There's a second trap. Another popular cut is
`astrometric_excess_noise < 1`, which removes stars whose positions scatter more than expected.
It cleans up the diagram even more. But stars with orbiting companions wobble, so they have extra
"noise" too. That cut would quietly throw away many of the binaries and planet hosts we're
looking for. Always ask whether a cut removes the thing you care about.

For the rest of this notebook, the background stars come from `clean`.

### 5.2 Famous stars on the HR diagram

Where is the Sun? It has $G_{BP} - G_{RP} \approx 0.82$ and $M_G \approx 4.67$.

**Try it.** Make the HR diagram of the clean sample again and mark the famous stars on it. The Sun
is already added for you. Look the famous stars up in `stars`, not `clean`, so that none go missing. Before you run it,
predict where Sirius B and 40 Eridani B will land.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.scatter(clean["bp_rp"], clean["abs_G"], s=1, alpha=0.2, color="gray")

for name, sid in famous.items():
    row = ____    # your code here
    ax.scatter(____, ____, s=30, color="red")    # your code here
    ax.text(____, ____, name, fontsize=8)    # your code here

ax.scatter(0.82, 4.67, marker="*", s=200, color="gold", edgecolor="k", label="Sun")

ax.invert_yaxis()
ax.set_xlabel(r"$G_{BP} - G_{RP}$ [mag]")
ax.set_ylabel(r"$M_G$ [mag]")
ax.legend()
plt.show()

## pandas summary

| Code | What it does |
|---|---|
| `pd.read_csv("file.csv")` | Load a table |
| `df.head()`, `df.describe()`, `len(df)` | First rows, summary statistics, number of rows |
| `df["col"]` | One column |
| `df["new"] = ...` | Add a column |
| `df[df["col"] > 5]` | Keep rows where a condition is true |
| `df.sort_values("col", ascending=False)` | Sort |
| `df.iloc[0]` | First row |
| `df["col"].isin(list)` | True where the value is in the list |

## Things to try on your own

- Repeat the HR diagram for stars within 20 pc only. Which features get clearer, and which get
  harder to see?
- Colour the 50 pc sky map by `bp_rp`. Are red and blue stars spread the same way across the sky?
- Use `parallax_error` and `phot_g_mean_mag` to plot how Gaia's precision depends on brightness.
  Use a log y-axis. Can you explain the shape? (There's a jump near $G = 13$. It comes from a
  change in how Gaia's camera reads out stars of different brightness.)
- Instead of the single ratio from section 2, plot the number of clean stars within distance $d$
  against $d$, on log-log axes. What slope do you expect? What do you get?
- Write your own ADQL query. For example: all stars within 100 pc brighter than G = 6.